In [5]:
from caida_as_org.fetch_data import build_as_org_snapshot

time = '20160701'
def build_org_helpers(as_info, org_info):
    def get_org_id(asn):
        info = as_info.get(asn)
        if info:
            return info["org_id"] if info["org_id"] != "" else info["opaque_id"]
            # return info["opaque_id"] if info["opaque_id"] != "" else info["org_id"]
        return str(asn)

    def get_org_name(org_id):
        # 先从 merged_org_info 取 name
        org_name = org_info.get(org_id, {}).get("name", "").strip()
        org_country = org_info.get(org_id, {}).get("country", "").strip()
        # 如果没有有效 name，就用 Org_<org_id> 回退
        if not org_name:
            org_name = f"Org_{org_id}"
        if not org_country:
            # 如果没有有效 country，就用 unknown
            org_country = f"Org_{org_id}"
        name = f"{org_name} ({org_country})"
        return name

    def from_same_org(a1, a2):
        return get_org_id(a1) == get_org_id(a2)

    def from_similar_org(a1, a2):
        def first_token(asn):
            oid = get_org_id(asn)
            # 拿第一个词，split 可能为空列表，做保护
            parts = get_org_name(oid).lower().split()
            return parts[0] if parts else oid.lower()
        return first_token(a1) == first_token(a2)

    return get_org_id, get_org_name, from_same_org, from_similar_org


merged_as_info, merged_org_info = build_as_org_snapshot(time, window=2)

Selected times: ['20160701', '20160401']
Processing Org file in 20160701
Processing Org file in 20160401


In [6]:
for item in merged_as_info.items():
    print(item)
    break

In [7]:
get_org_id, get_org_name, from_same_org, from_similar_org = build_org_helpers(merged_as_info, merged_org_info)

In [8]:
asn = '3356'

print(get_org_id(asn))
print(get_org_name(get_org_id(asn)))


3356
LEVEL3 (LVLT-ARIN)


In [1]:
neighbor_prompt = "9 Neighbors for AS3356 (6669 in total): AS3356's Customer (downstream neighbor): AS199118 from Heinlein-Support GmbH (Country: DE); AS3356's Customer (downstream neighbor): AS38022 from Research and Education Advanced Network New Zealand (Country: NZ); AS3356's Customer (downstream neighbor): AS2159 from HP Inc. (Country: US); AS3356's Customer (downstream neighbor): AS399716 from ISHIP, INC. (Country: US); AS3356's Customer (downstream neighbor): AS1712 from Renater (Country: FR); AS3356's Customer (downstream neighbor): AS200520 from Procter & Gamble Service GmbH (Country: DE); AS3356's Customer (downstream neighbor): AS1736 from Marquette University (Country: US); AS3356's Customer (downstream neighbor): AS19834 from Conversant, LLC (Country: US); AS3356's Customer (downstream neighbor): AS201284 from Devexperts GmbH (Country: DE); Following that, there will be 0 additional neighbors’ information for AS3356."

import re
def get_neighbor_spans(prompt: str, asn: str):
    """
    在 prompt 中定位所有以 "AS{asn}'s" 开头、以 ';' 结尾的邻居描述块，
    返回它们的 (start, end) 字符区间列表，顺序即 prompt 里的顺序。
    """
    pattern = re.escape(f"AS{asn}'s")
    spans = []
    for m in re.finditer(pattern, prompt):
        st = m.start()
        semi = prompt.find(";", st)
        if semi < 0:
            semi = len(prompt)
        spans.append((st, semi + 1))
    return spans

span = get_neighbor_spans(neighbor_prompt, "3356")

In [5]:
print(span)
# print(neighbor_prompt[span[0][0]:span[0][1]])
neighbor_text = [neighbor_prompt[st:ed] for st, ed in span]
neighbor_asn = []
for neighbor in neighbor_text:
    match = re.search(r': AS(\d+)', neighbor)
    if match: neighbor_asn.append(match.group(1))
    else: neighbor_asn.append("N/A")
print(neighbor_asn)


[(40, 131), (132, 252), (253, 328), (329, 410), (411, 486), (487, 586), (587, 675), (676, 760), (761, 846)]
['199118', '38022', '2159', '399716', '1712', '200520', '1736', '19834', '201284']
